# **Set Up**

In [ ]:
# Installations
%%capture
!pip install --pre deepchem
!pip install rdkit-pypi
!pip install selfies
!pip install PyYAML
!pip install -U PyYAML

import sys, os
import yaml

import deepchem as dc
import pandas as pd
import tensorflow as tf
import numpy as np
from numpy.testing import assert_allclose

import selfies as sf
from rdkit import Chem
import rdkit, rdkit.Chem, rdkit.Chem.Draw
from rdkit.Chem.Draw import IPythonConsole

from keras.models import Sequential, load_model
from keras.layers import Dense
from keras.layers import Dropout
from keras.layers import LSTM
from keras.callbacks import ModelCheckpoint
#from keras.utils import np_utils

from google.colab import files
from google.colab import drive

drive.mount('/content/gdrive')

In [ ]:
# Prepare associated zip folder of model documentation 'Molecule-RNN-main.zip'
!unzip -o ./gdrive/MyDrive/Molecule-RNN-main > /dev/null

# **RNN-based Generation of Promoieties Directly onto Starting Drug Structures**

In [ ]:
# Adjustable variables
model_type = 'LSTM' # GRU or LSTM
out_file_name = 'LSTM_generated_excipients'
batch_size = 2048 # Default 2048, input number 1 or greater
num_batches = 20 # Default 20, input number 1 or greater
dataset_dir = './Molecule-RNN-main/dataset/molgan_input.smi'
vocab_path = './Molecule-RNN-main/vocab/selfies_merged_vocab.yaml'

#dataset_dir = './Molecule-RNN-main/dataset/molgan_input.smi'
#vocab_path = './Molecule-RNN-main/vocab/selfies_merged_vocab.yaml'

with open('Molecule-RNN-main/train.yaml') as f: # Possibly adjust pathway for local runs based on folder location
  list_doc = yaml.safe_load(f)
  list_doc['rnn_config']['rnn_type'] = model_type
  list_doc['out_file_name'] = out_file_name
  list_doc['batch_size'] = batch_size
  list_doc['num_batches'] = num_batches
  list_doc['dataset_dir'] = dataset_dir
  list_doc['vocab_path'] = vocab_path
  with open('Molecule-RNN-main/train.yaml', "w") as f: # Possibly adjust pathway for local runs based on folder location
    yaml.dump(list_doc, f)

In [ ]:
# Enter your API (in SMILES code) here to build directly off of this structure. Please press enter when done typing.
# Note: The last character of the SMILES string is where the promoiety will begin
# Possible start for removal is Carboxylic acid: O=C(O)C or carbon dioxide, CO
# Cholic acid: CC(C1CCC2C1(C)C(O)CC3C2C(O)CC4C3(C)CCC(O)C4)CCC(=O)O
# Benzoic acid: C1=CC=C(C=C1)C(=O)O
# NOTE: A 1 Character drug is not legal

with open('/content/Molecule-RNN-main/train.yaml') as f:
    list_doc = yaml.safe_load(f)
    while True:
      try:
        drug = sf.encoder(input('Please enter your DRUG in SMILES format: ').replace('\\\\','\\'))
        drug = drug.replace('\\','*')
        list_doc['drug'] = drug.replace('\\','*')
        print('Your encoded drug is: ' + str(drug))
        with open('/content/Molecule-RNN-main/train.yaml', "w") as f:
          yaml.dump(list_doc, f)
        print('Success!')
        break
      except:
        print('This is not a valid SMILES code; try again.')
        continue

Please enter your DRUG in SMILES format: CO
Your encoded drug is: [C][O]
Success!


In [ ]:
# Model Training
!python ./Molecule-RNN-main/train.py

device:  cuda
number of workers to load data:  2
which vocabulary to use:  selfies
total number of SMILES loaded:  2256
total number of valid SELFIES:  2256
/usr/local/lib/python3.10/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
{75: '<eos>', 77: '<pad>', 76: '<sos>', 42: '[#Branch1]', 65: '[#Branch2]', 40: '[#C-1]', 69: '[#C]', 59: '[#N+1]', 45: '[#N]', 58: '[=As]', 35: '[=Branch1]', 4: '[=Branch2]', 30: '[=C]', 33: '[=N+1]', 38: '[=N-1]', 41: '[=NH1+1]', 49: '[=NH2+1]', 19: '[=N]', 72: '[=O+1]', 1: '[=O]', 62: '[=P+1]', 61: '[=PH1]', 0: '[=P]', 25: '[=Ring1]', 43: '[=Ring2]', 31: '[=S+1]', 47: '[=SH1]', 13: '[=S]', 11: '[=Se+1]', 24: '[=Se]', 7: '[=Te+1]', 27: '[Al]', 74: '[As]', 71: '[B-1]', 67: '[BH1-1]', 8: '[BH2-1]', 10: '[BH3-1]', 60: '[B]', 56: '[Br]', 23: '[Branch1]', 70: '[Branch2]', 32: '[C+1]', 34: '[C-1]', 29: '[CH1-1]', 14: '[C]', 57: '[Cl+3]', 18: '[Cl

In [ ]:
# Generate prodrugs
# Output file is located in the results folder with provided name
!python ./Molecule-RNN-main/sample.py

device:  cuda
/content/./Molecule-RNN-main/sample.py:60: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(
100% 20/20 [00:18<00:00,  1.10it/s]
